# Baby Shower Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.14


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.


## Problem Statement

Book problem `2.14` is a binary invitation problem for a baby shower. Each guest is associated with
the value of the gift they would bring, and the invitation list must satisfy several logical rules:

- José attends only if Valentina attends,
- José does not attend if both Ana and Daniel attend,
- Ana and Camila are complements: exactly one of them attends,
- Ana attends only if Daniel and Valentina attend together.


In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

try:
    from amplpy import AMPL
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install amplpy in the active environment before running this notebook.") from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


GUESTS = [1, 2, 3, 4, 5]
NAMES = {1: "Jose", 2: "Ana", 3: "Daniel", 4: "Valentina", 5: "Camila"}
GIFT_VALUE = {1: 62, 2: 30, 3: 85, 4: 82, 5: 77}


@timed
def solve_baby_with_ampl(solver: str = "highs"):
    ampl = AMPL()
    ampl.eval(f"option solver {solver};")
    ampl.eval(
        r'''
        set I ordered;
        param U {I};
        var x {I} binary;

        maximize TotalValue:
            sum {i in I} U[i] * x[i];

        subject to JoseNeedsValentina:
            x[1] <= x[4];

        subject to JoseNotWithAnaAndDaniel:
            x[1] <= 2 - x[2] - x[3];

        subject to AnaVsCamila:
            x[2] = 1 - x[5];

        subject to AnaNeedsDanielAndValentina:
            x[4] + x[3] <= 1 + x[2];

        subject to AnaNeedsValentina:
            x[2] <= x[4];

        subject to AnaNeedsDaniel:
            x[2] <= x[3];
        '''
    )
    ampl.set["I"] = GUESTS
    ampl.param["U"] = GIFT_VALUE
    ampl.solve(solver=solver)
    values = ampl.get_variable("x").get_values().to_dict()
    invited = [guest for guest in GUESTS if round(values[guest]) == 1]
    return {"invited": invited, "names": [NAMES[guest] for guest in invited], "total_value": int(round(ampl.get_value("TotalValue")))}


## Book Model In AMPL


In [2]:
result, elapsed = solve_baby_with_ampl()
print("AMPL result:", result)
print(f"Elapsed time: {elapsed:.6f} seconds")
assert result["invited"] == [1, 4, 5]
assert result["total_value"] == 221


HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 221
0 simplex iterations
1 branching nodes
AMPL result: {'invited': [1, 4, 5], 'names': ['Jose', 'Valentina', 'Camila'], 'total_value': 221}
Elapsed time: 0.014908 seconds
